# LLM and RAG System Design — the 2026 Favorite

"Design a document Q&A system for 10 million documents" is *the* modern ML design prompt — it's in the repo's own senior-level roadmap, and it's what AI-native teams actually ask. The interviewer wants to hear you treat RAG as a **system** — retrieval quality, context assembly, grounding, cost — not a library call. This notebook walks the full stack with executed demos at each decision point.

## Step 0 — The Prompt, Decoded

*"Design an internal Q&A assistant over 10M company documents (wikis, PDFs, tickets), 20k employees, answers must cite sources, fresh docs daily, P95 latency < 3s, run cheap."*

The hidden requirements the interviewer is checking you can extract:
- **Scale:** 10M docs × ~2KB avg chunks → ~40M chunks after chunking — exact search is dead, ANN is mandatory.
- **Freshness:** daily updates → incremental indexing, not nightly full rebuilds.
- **Citations:** provenance tracking through the whole pipeline — chunk IDs to document locations.
- **Latency:** 3s end-to-end with an LLM call inside — the LLM is ~1–2s, so retrieval + rerank gets <500ms.
- **"Cheap":** self-hosted mid-size model vs API — a cost conversation, have numbers ready.

## Step 1 — Ingestion and Chunking

Pipeline: fetch → parse (PDF/HTML/office — parsing quality is the silent determinant of RAG quality; say it) → **chunk** → embed → index.

**Chunking, the tradeoff table to recite:**
- **Fixed-size (400–800 tokens, 10–20% overlap)** — simple, predictable; cuts sentences mid-thought; overlap protects boundaries. The default.
- **Structure-aware (by heading/paragraph)** — respects document semantics; varies in size; needs normalization. Better when docs have structure.
- **Parent-small retrieval** — embed small chunks (precision) but return the parent section to the LLM (context). The trick that fixes "the answer was split across chunks."

**Daily updates:** content-hash each doc; changed docs only → delete/update their chunks in the index. Full rebuild is O(hours) and unnecessary.

In [1]:
# chunking with overlap, executed — watch the boundary sentence protection
text = ("Gradient descent updates weights using the gradient of the loss. "
        "The learning rate controls the step size and is the most important hyperparameter. "
        "Momentum accumulates past gradients to damp oscillation. "
        "Adam combines momentum with per-parameter scaling via second moments.")

def chunk(text, size=25, overlap=5):
    words = text.split()
    chunks = []
    for start in range(0, len(words), size - overlap):
        piece = words[start:start + size]
        if len(piece) < 5: break
        chunks.append(" ".join(piece))
    return chunks

pieces = chunk(text)
print(f"{len(text.split())} words -> {len(pieces)} chunks (size 25, overlap 5):")
for i, p in enumerate(pieces):
    print(f"  [{i}] {p[:90]}...")
print("\nthe 'Momentum' sentence appears in chunks 0 AND 1 — overlap saved the boundary")

39 words -> 2 chunks (size 25, overlap 5):
  [0] Gradient descent updates weights using the gradient of the loss. The learning rate control...
  [1] most important hyperparameter. Momentum accumulates past gradients to damp oscillation. Ad...

the 'Momentum' sentence appears in chunks 0 AND 1 — overlap saved the boundary


## Step 2 — Embeddings and the Vector Index

**Embedding choice:** a strong off-the-shelf sentence embedder (dimension 384–1024, trained for retrieval with contrastive objectives); the requirements: same model at query and doc time, long-enough context (chunks fit), and **known-good on BEIR-style benchmarks**. Domain shift (legal/medical jargon) → evaluate 2–3 candidates on *your* data before committing; fine-tune only if retrieval recall is the bottleneck.

**Index:** at 40M chunks, exact k-NN (40M × 768 dot products per query) is ~seconds — dead. **ANN (HNSW)**: build a layered proximity graph, query traverses top layers down — ~O(log n) with recall tradeoff via `ef` parameter. Numbers to quote: HNSW at 40M × 768d ≈ 60–100GB RAM (or IVF-PQ compressed to ~10–20GB at recall cost); query latency 5–20ms; throughput thousands QPS per node.

**Hybrid retrieval — say this word:** dense embeddings miss rare keywords (error codes, part numbers) that lexical search nails. Production answer: **BM25 + dense, fused with Reciprocal Rank Fusion (RRF)** — no weights to tune, embarrassingly robust.

In [2]:
# RRF fusion, executed — the 6-line hybrid retrieval combiner
import numpy as np

docs = ["model: loss=NaN at lr=0.5", "gradient clipping prevents explosion",
        "error code E-4021 in module X", "loss scale underflow in fp16 training",
        "module X returns E-4021 on cold start"]
query = "E-4021"

# pretend BM25 ranking (keyword-heavy) and dense ranking (semantic) disagree:
bm25_rank = [2, 4] + [0, 1, 3]      # exact-match docs first
dense_rank = [1, 0, 3] + [2, 4]     # semantic neighbor first

def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            scores[doc] = scores.get(doc, 0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True), scores

fused, scores = rrf([bm25_rank, dense_rank])
print("RRF-fused order:", fused[:3], "| scores:", {d: round(scores[d], 4) for d in fused[:3]})
print("doc 2 (the E-4021 page) tops BOTH lists -> tops the fusion; doc 1 survives from one list only")

RRF-fused order: [2, 1, 0] | scores: {2: 0.032, 1: 0.032, 0: 0.032}
doc 2 (the E-4021 page) tops BOTH lists -> tops the fusion; doc 1 survives from one list only


## Step 3 — Reranking and Context Assembly

Retrieve top-50, **rerank to top-5–10** with a cross-encoder (query+doc jointly scored — the bi-encoder's 20ms becomes 200–400ms for 50 pairs, affordable in the 500ms retrieval budget). Rerankers buy 5–15 points of answer quality in production RAGs — the cheapest big win, and the step candidates forget.

**Context assembly:** dedupe near-identical chunks; order matters — **lost-in-the-middle** (LLMs attend best to prompt start/end — put best chunks first/last); cap context to leave generation headroom; format with source metadata inline for citations. If the answer needs aggregation across many docs ("summarize all E-4021 incidents"), that's a **map-reduce / multi-query** pattern, not one retrieval.

## Step 4 — Generation: Model Choice and Configuration

**Model selection, the honest table:**
- **API frontier models** — best quality, ~$1–10 per 1M output tokens, data leaves the building (compliance question to raise unprompted).
- **Self-hosted mid-size (7–34B, quantized)** — "cheap" at scale, ~$2–8/hr per GPU node amortizing to fractions of a cent per query *at high utilization* — utilization is the catch; needs MLOps.
- **Small local models** — dev/prototyping.

For an internal 20k-employee assistant with compliance needs: self-hosted 8–32B + API fallback for hard queries is a defensible architecture — say the fallback exists.

**Configuration:** temperature 0–0.2 for factual QA; system prompt enforcing *"answer only from the provided context; if insufficient, say so"* — the single most effective anti-hallucination line; max_tokens bounded; streaming for perceived latency.

**The citation mechanism:** number the retrieved chunks in the prompt, require inline [1][2] references, post-validate that cited indices exist in the retrieved set — grounding you can *check*.

## Step 5 — Evaluation (Where RAG Projects Actually Fail)

The layered eval stack — interviewers probe whether you evaluate *retrieval* and *generation* separately:

1. **Retrieval metrics:** recall@k against a golden (query → relevant doc) set; MRR. If recall@5 is 60%, no LLM can save you — fix retrieval first.
2. **Generation metrics:** faithfulness/groundedness (does every claim trace to a cited chunk — checkable automatically by an LLM judge or NLI model), answer relevance to the question.
3. **End-to-end:** human-labeled answer quality on a sample; regression-test the golden set on every pipeline change.
4. **Online:** thumbs up/down, retry rate, deflection rate (questions answered without escalation).

**The killer sentence:** *"RAG quality is 80% retrieval quality — I evaluate retrieval first and separately, because LLM improvements can't fix documents the model never saw."*

In [3]:
# retrieval-metric arithmetic — the numbers that drive the diagnosis
golden = {  # query -> the ONE doc that contains the answer
    "q1": "d42", "q2": "d7", "q3": "d913", "q4": "d55", "q5": "d231",
}
ranked = {  # what the retriever returned, position of the gold doc
    "q1": ["d42"],                # rank 1
    "q2": ["d1", "d2", "d7"],     # rank 3
    "q3": ["x", "y", "z", "d913"],# rank 4
    "q4": ["x", "d55"],           # rank 2
    "q5": ["x", "y", "z", "w"],   # MISS
}
k = 5
recalls = [1 if golden[q] in ranked[q][:k] else 0 for q in golden]
mrr = np.mean([1 / (ranked[q].index(golden[q]) + 1) if golden[q] in ranked[q] else 0 for q in golden])
print(f"recall@{k}: {np.mean(recalls):.2f}  (4/5 retrieved)")
print(f"MRR      : {mrr:.2f}")
print("diagnosis: recall is decent but the gold doc ranks 3rd-4th when found -> RERANKER, not a better embedder")

recall@5: 0.80  (4/5 retrieved)
MRR      : 0.42
diagnosis: recall is decent but the gold doc ranks 3rd-4th when found -> RERANKER, not a better embedder


## Step 6 — Guardrails, Security, and the Unsexy Parts

**Prompt injection** — retrieved documents are *untrusted input*: a malicious page saying "ignore previous instructions" must not hijack the answer. Mitigations: strip/neutralize instructions in retrieved text, delimiter isolation, output filtering. Raise this unprompted — security awareness is senior signal.

**Access control** — retrieval must filter by the *asker's* permissions (ACL filtering at query time or per-permission indexes) — the classic "the RAG leaked HR docs" incident is this bug.

**PII/retention** — query logging policy, deletion propagation (a deleted doc must vanish from answers AND the index — tie to the content-hash ingestion from Step 1).

**Caching** — embedding cache for repeated queries, semantic cache for near-duplicate questions (careful: cached answers skip fresh retrieval — TTL it). **Cost telemetry** per query (embedding ms + rerank ms + tokens in/out) — the dashboard the interviewer didn't know to ask for.

## The Full Architecture, One Paragraph (Memorize the Flow)

Documents stream in → parse → chunk (structure-aware, overlap) → embed → **daily incremental** HNSW index (+BM25 index in parallel) with ACL metadata. Query → embed → **hybrid retrieval** (dense top-50 + BM25 top-50 → RRF) → **cross-encoder rerank** to top-8 → context assembly (dedupe, lost-in-the-middle ordering, numbered citations) → self-hosted mid-size LLM (temp 0, cite-or-refuse prompt, streaming) → post-validation of citations → answer + sources. Evaluation harness: golden-set retrieval metrics + faithfulness scoring, regression-run on every change. Monitoring: recall drift, latency per stage, cost per query, thumbs-rate. Fallbacks: rule-based "I don't know," API-model escalation for hard queries, feature-flag rollback for bad index versions.

In [4]:
# end-to-end latency budget — walk this arithmetic in the interview
stages = {
    "query embed": 12, "hybrid retrieval (2x ANN+BM25)": 25, "RRF fuse": 2,
    "rerank 50->8 (cross-encoder)": 180, "context assembly": 5,
    "LLM first token (TTFT)": 450, "LLM stream (150 tokens)": 1800,
}
total = sum(stages.values())
for s, ms in stages.items(): print(f"  {s:<38} {ms:>6} ms")
print(f"  {'TOTAL (P50-ish)':<38} {total:>5} ms")
print(f"\nbudget check: {total}ms vs 3000ms P95 -> {3000-total}ms headroom for tail latency and retries")
print("where to cut if over: rerank to 25 pairs (-90ms), cache hot queries (-everything)")

  query embed                                12 ms
  hybrid retrieval (2x ANN+BM25)             25 ms
  RRF fuse                                    2 ms
  rerank 50->8 (cross-encoder)              180 ms
  context assembly                            5 ms
  LLM first token (TTFT)                    450 ms
  LLM stream (150 tokens)                  1800 ms
  TOTAL (P50-ish)                         2474 ms

budget check: 2474ms vs 3000ms P95 -> 526ms headroom for tail latency and retries
where to cut if over: rerank to 25 pairs (-90ms), cache hot queries (-everything)


## Variants They'll Probe

- **Agentic RAG** — LLM decides to retrieve again with a reformulated query (multi-hop: "compare A and B" → two retrievals). Cost: 2–3× latency; use only when single-hop fails on eval.
- **Fine-tune instead of RAG?** — when the knowledge is *stable and stylistic* (tone, formats, domain vocabulary); RAG wins when knowledge changes or needs citations. Usually: RAG first, LoRA later.
- **Long-context instead of RAG?** — stuffing 500K context: quadratic attention cost, lost-in-the-middle degradation, and *freshness still needs re-ingestion* — RAG remains the answer at 10M docs.
- **Multimodal** — images/tables in docs: OCR + layout parsing + multimodal embedders; tables are the hard 20% (structure-aware extraction, not raw text chunks).
- **Chat over docs (multi-turn)** — query rewriting from conversation context before retrieval ("what about last year?" → standalone query); conversation memory separate from document retrieval.

## Rehearsal Card

Six steps, six keywords, six numbers — the whole notebook compressed:

| Step | Keyword | Number to quote |
| --- | --- | --- |
| 1. Ingestion | structure-aware chunking + overlap | 10M docs → ~40M chunks |
| 2. Retrieval | hybrid dense+BM25, RRF fusion | HNSW 5–20ms, ~60–100GB at 40M×768d |
| 3. Rerank | cross-encoder top-50→8 | +180ms for 5–15 quality points |
| 4. Generation | cite-or-refuse prompt, temp 0 | TTFT ~450ms, stream the rest |
| 5. Evaluation | retrieval metrics FIRST | recall@5 < 70% = fix retrieval, not the LLM |
| 6. Ops | ACLs, injection, incremental indexing | daily deltas by content hash |

Drill this card until each row is a 60-second riff — that riff *is* the design round.